# Statistical Modeling — Tutorial

**Primary outcome:** Build, compare, and interpret statistical models — from polynomial regression to logistic regression — and apply regularization to prevent overfitting.

**Time:** 60–90 minutes

---

## What You Will Learn

| Section | Topic |
|---------|-------|
| 1 | The Modeling Mindset |
| 2 | Baseline: Simple Linear Regression |
| 3 | Model Selection Framework |
| 4 | Polynomial Regression in Practice |
| 5 | Logistic Regression: Binary Classification |
| 6 | Understanding Logistic Regression |
| 7 | Regularization: L1 and L2 |
| 8 | Comparing Models with Cross-Validation |
| 9 | Model Interpretation |
| 10 | Putting It Together: Model Report |
| 11 | Practice Exercises |

## Section 1 — The Modeling Mindset

### What Is a Statistical Model?

A statistical model is a mathematical description of the relationship between variables. Given input features **X**, we want to estimate an output **y**:

```
y = f(X) + ε
```

- `f(X)` is the **signal** — the true pattern we want to learn.
- `ε` is the **noise** — random variation we cannot and should not try to fit.

### The Modeling Pipeline

```
  ┌─────────┐    ┌────────┐    ┌─────┐    ┌──────────┐    ┌───────────┐
  │ Prepare │───▶│ Select │───▶│ Fit │───▶│ Evaluate │───▶│ Interpret │
  └─────────┘    └────────┘    └─────┘    └──────────┘    └───────────┘
  Clean data,    Choose model   Train on   Measure on    Explain what
  split train/   complexity     train set  test set      coefficients
  test                                                    mean
```

### Overfitting vs. Underfitting

```
Error
  │
  │  ╲                         Underfitting: model too simple
  │   ╲   Train error          Overfitting:  model too complex
  │    ╲  ___________________  Ideal:        sweet spot in the middle
  │     ╲/
  │      ╲
  │  ─────╲────────────────── Test error
  │        ╲____/‾‾‾‾‾‾‾‾‾‾‾
  └──────────────────────────▶ Model Complexity
     Under  Ideal  Over
```

**Key insight:** A model that memorises the training data perfectly will fail on new data. We want the model to *generalise*.

In [ ]:
# ── Imports and global settings ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    r2_score, mean_squared_error,
    accuracy_score, classification_report,
    roc_auc_score, confusion_matrix
)
from sklearn.datasets import load_breast_cancer
from sklearn.inspection import permutation_importance

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully.')

## Section 2 — Baseline: Simple Linear Regression

Before trying anything fancy, always start with the simplest possible model. This gives you a **baseline** to beat.

We will use the classic **Tips dataset** from seaborn: restaurant bill data with 244 rows and columns including `total_bill`, `tip`, `sex`, `smoker`, `day`, `time`, and `size`.

**Goal:** Predict `tip` from `total_bill`.

In [ ]:
# Load and inspect the tips dataset
tips = sns.load_dataset('tips')
print(f'Shape: {tips.shape}')
print(f'\nFirst 5 rows:')
tips.head()

In [ ]:
# Prepare data for simple linear regression
X_tips = tips[['total_bill']].values
y_tips = tips['tip'].values

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_tips, y_tips, test_size=0.2, random_state=42
)

# Fit the baseline linear regression
baseline = LinearRegression()
baseline.fit(X_train_t, y_train_t)

y_pred_baseline = baseline.predict(X_test_t)
r2_baseline = r2_score(y_test_t, y_pred_baseline)

print('── Baseline Model ──────────────────────────────────')
print(f'  Intercept  : {baseline.intercept_:.4f}')
print(f'  Coefficient: {baseline.coef_[0]:.4f}')
print(f'  Test R²    : {r2_baseline:.4f}')
print()
print('Interpretation:')
print(f'  For every $1 increase in the total bill, the expected tip')
print(f'  increases by ${baseline.coef_[0]:.2f}.')
print(f'  When the bill is $0, the predicted tip is ${baseline.intercept_:.2f} (baseline offset).')

In [ ]:
# Scatter plot + regression line
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: fit
ax = axes[0]
ax.scatter(X_test_t, y_test_t, alpha=0.6, label='Actual', color='steelblue')
x_line = np.linspace(X_tips.min(), X_tips.max(), 200).reshape(-1, 1)
ax.plot(x_line, baseline.predict(x_line), color='tomato', linewidth=2, label='Fit')
ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Tip ($)')
ax.set_title(f'Simple Linear Regression  (Test R² = {r2_baseline:.3f})')
ax.legend()

# Right: residual plot
residuals = y_test_t - y_pred_baseline
ax = axes[1]
ax.scatter(y_pred_baseline, residuals, alpha=0.6, color='darkorange')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Predicted Tip ($)')
ax.set_ylabel('Residual')
ax.set_title('Residual Plot')

plt.tight_layout()
plt.show()

print('Residual plot tip: residuals should be randomly scattered around 0.')
print('A curved pattern would suggest a non-linear relationship.')

## Section 3 — Model Selection Framework

How do we decide which model is best? The key principle:

> **Fit on training data. Evaluate on test data.**

We will fit linear (degree=1), quadratic (degree=2), cubic (degree=3), and highly flexible (degree=10) polynomial models on a synthetic sine-wave dataset, and compare train vs. test R².

In [ ]:
# Synthetic sine-curve dataset
np.random.seed(42)
X_sin = np.linspace(0, 2 * np.pi, 80).reshape(-1, 1)
y_sin = np.sin(X_sin).ravel() + np.random.normal(0, 0.25, 80)

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(
    X_sin, y_sin, test_size=0.25, random_state=42
)

degrees = [1, 2, 3, 10]
colors = ['steelblue', 'seagreen', 'darkorange', 'firebrick']

fig, axes = plt.subplots(1, 4, figsize=(17, 4), sharey=True)

results = []
x_plot = np.linspace(0, 2 * np.pi, 300).reshape(-1, 1)

for ax, deg, col in zip(axes, degrees, colors):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    pipe.fit(X_tr_s, y_tr_s)

    train_r2 = pipe.score(X_tr_s, y_tr_s)
    test_r2  = pipe.score(X_te_s, y_te_s)
    results.append({'degree': deg, 'train_r2': train_r2, 'test_r2': test_r2})

    ax.scatter(X_te_s, y_te_s, alpha=0.5, s=20, color='grey', label='Test data')
    ax.plot(x_plot, pipe.predict(x_plot), color=col, linewidth=2)
    ax.set_title(f'Degree {deg}\nTrain R²={train_r2:.2f}  Test R²={test_r2:.2f}', fontsize=9)
    ax.set_xlabel('X')

axes[0].set_ylabel('y')
plt.suptitle('Polynomial Fits on Sine Curve — Underfitting to Overfitting', y=1.02)
plt.tight_layout()
plt.show()

print('\n── Train vs. Test R² ─────────────────────────────────────')
print(f'  {"Degree":>8}  {"Train R²":>10}  {"Test R²":>10}')
for r in results:
    flag = '  ← OVERFIT' if r['train_r2'] - r['test_r2'] > 0.2 else ''
    print(f'  {r["degree"]:>8}  {r["train_r2"]:>10.4f}  {r["test_r2"]:>10.4f}{flag}')

### Takeaway

| Degree | Behaviour |
|--------|-----------|
| 1 | **Underfitting** — too rigid, high bias |
| 2–3 | **Good fit** — low bias, low variance |
| 10 | **Overfitting** — memorises noise, poor test R² |

The **train R²** of degree=10 looks great, but the **test R²** is poor — this is the classic overfitting trap. Always check test performance.

## Section 4 — Polynomial Regression in Practice

Back to the Tips dataset. Is the `total_bill → tip` relationship truly linear?

We will:
1. Inspect residuals from the linear model.
2. Fit degree=2 polynomial regression.
3. Compare test R² for degree=1 and degree=2.

In [ ]:
# Visual comparison: linear vs. quadratic fit on tips
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

x_range = np.linspace(X_tips.min(), X_tips.max(), 300).reshape(-1, 1)

for ax, deg, title in zip(
    axes[:2],
    [1, 2],
    ['Linear (degree=1)', 'Quadratic (degree=2)']
):
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    pipe.fit(X_train_t, y_train_t)
    test_r2 = pipe.score(X_test_t, y_test_t)

    ax.scatter(X_tips, y_tips, alpha=0.3, s=18, color='steelblue')
    ax.plot(x_range, pipe.predict(x_range), color='tomato', linewidth=2)
    ax.set_xlabel('Total Bill ($)')
    ax.set_ylabel('Tip ($)')
    ax.set_title(f'{title}\nTest R² = {test_r2:.4f}')

# Residuals comparison
lin_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=1, include_bias=False)),
    ('reg',  LinearRegression())
])
quad_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('reg',  LinearRegression())
])
lin_pipe.fit(X_train_t, y_train_t)
quad_pipe.fit(X_train_t, y_train_t)

res_lin  = y_test_t - lin_pipe.predict(X_test_t)
res_quad = y_test_t - quad_pipe.predict(X_test_t)

ax = axes[2]
ax.scatter(lin_pipe.predict(X_test_t),  res_lin,  alpha=0.5, label='Linear',    color='steelblue', s=20)
ax.scatter(quad_pipe.predict(X_test_t), res_quad, alpha=0.5, label='Quadratic', color='seagreen',  s=20, marker='^')
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Predicted Tip ($)')
ax.set_ylabel('Residual')
ax.set_title('Residuals: Linear vs. Quadratic')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Systematic degree search using sklearn Pipeline + CV
print('── Polynomial Degree Search (5-fold CV on full tips data) ──────────')
print(f'  {"Degree":>8}  {"CV R² Mean":>12}  {"CV R² Std":>12}')

for deg in range(1, 7):
    pipe = Pipeline([
        ('poly',  PolynomialFeatures(degree=deg, include_bias=False)),
        ('reg',   LinearRegression())
    ])
    scores = cross_val_score(pipe, X_tips, y_tips, cv=5, scoring='r2')
    print(f'  {deg:>8}  {scores.mean():>12.4f}  {scores.std():>12.4f}')

print()
print('Conclusion: The relationship is largely linear. Degree=1 and degree=2')
print('yield similar CV scores. Higher degrees do NOT improve generalisation.')

## Section 5 — Logistic Regression: Binary Classification

Linear regression predicts a **continuous** value. For binary outcomes (yes/no, malignant/benign), we use **logistic regression**.

We will use the **Breast Cancer Wisconsin dataset** (sklearn built-in):
- 569 samples
- 30 numeric features (cell measurements)
- Target: `0 = malignant`, `1 = benign`

In [ ]:
# Load breast cancer dataset
bc = load_breast_cancer()
X_bc = pd.DataFrame(bc.data, columns=bc.feature_names)
y_bc = bc.target

print('── Breast Cancer Dataset ────────────────────────────────────')
print(f'  Samples  : {X_bc.shape[0]}')
print(f'  Features : {X_bc.shape[1]}')
print(f'  Classes  : {bc.target_names.tolist()}')
print()
counts = pd.Series(y_bc).value_counts().sort_index()
for label, count in zip(bc.target_names, counts):
    print(f'  {label:>10}: {count:>4} ({count/len(y_bc)*100:.1f}%)')

print('\nFirst 5 rows (selected features):')
X_bc[bc.feature_names[:6]].head()

In [ ]:
# EDA: correlation heatmap of top 10 features
top10_features = X_bc.corr()['mean radius'].abs().nlargest(10).index.tolist()
corr = X_bc[top10_features].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr, annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, vmin=-1, vmax=1
)
plt.title('Correlation Heatmap — Top 10 Features (by correlation with mean radius)')
plt.tight_layout()
plt.show()

In [ ]:
# Train/test split and fit logistic regression
X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

# Scale features (important for logistic regression)
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr_bc)
X_te_scaled  = scaler.transform(X_te_bc)

log_reg = LogisticRegression(max_iter=10_000, random_state=42)
log_reg.fit(X_tr_scaled, y_tr_bc)

y_pred_bc = log_reg.predict(X_te_scaled)
acc = accuracy_score(y_te_bc, y_pred_bc)

print(f'Test Accuracy: {acc:.4f}  ({acc*100:.1f}%)')
print()
print(classification_report(y_te_bc, y_pred_bc, target_names=bc.target_names))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_te_bc, y_pred_bc)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=bc.target_names, yticklabels=bc.target_names
)
plt.title('Confusion Matrix — Logistic Regression')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Section 6 — Understanding Logistic Regression

Logistic regression models the **probability** of a class using the sigmoid function:

$$P(y=1 | X) = \sigma(w^T X + b) = \frac{1}{1 + e^{-(w^T X + b)}}$$

The output is always between 0 and 1 — a valid probability.

**Odds ratio interpretation:**  
If a coefficient is `w`, a 1-unit increase in that feature multiplies the **odds** of the positive class by `e^w`.

In [ ]:
# Visualise the sigmoid function
z = np.linspace(-8, 8, 300)
sigma = 1 / (1 + np.exp(-z))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(z, sigma, color='steelblue', linewidth=2)
ax.axhline(0.5, color='tomato', linestyle='--', linewidth=1, label='Decision boundary (p=0.5)')
ax.axvline(0,   color='grey',   linestyle=':',  linewidth=1)
ax.fill_between(z, sigma, 0.5, where=(sigma > 0.5), alpha=0.1, color='seagreen',  label='Predict benign')
ax.fill_between(z, sigma, 0.5, where=(sigma < 0.5), alpha=0.1, color='tomato',    label='Predict malignant')
ax.set_xlabel('Linear score z = wX + b')
ax.set_ylabel('Probability')
ax.set_title('Sigmoid / Logistic Function')
ax.legend(fontsize=8)

# Probability of malignancy vs. mean radius (one feature)
feature_idx = list(X_bc.columns).index('mean radius')
radius_range = np.linspace(X_bc['mean radius'].min(), X_bc['mean radius'].max(), 300)

# Build a single-feature scaler+logistic for illustration
X_radius = X_bc[['mean radius']].values
sc1 = StandardScaler()
X_r_scaled = sc1.fit_transform(X_radius)
lr1 = LogisticRegression(max_iter=1000)
lr1.fit(X_r_scaled, y_bc)

r_plot = sc1.transform(radius_range.reshape(-1, 1))
proba  = lr1.predict_proba(r_plot)[:, 1]

ax = axes[1]
# Jitter actual labels for visibility
jitter = np.random.uniform(-0.02, 0.02, len(y_bc))
ax.scatter(X_bc['mean radius'], y_bc + jitter, alpha=0.15, s=10,
           c=['tomato' if yi == 0 else 'steelblue' for yi in y_bc])
ax.plot(radius_range, proba, color='darkorange', linewidth=2.5, label='P(benign)')
ax.axhline(0.5, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Mean Radius')
ax.set_ylabel('Probability of being Benign')
ax.set_title('P(Benign) vs. Mean Radius')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Odds ratios for the top 5 most influential coefficients
coef_series = pd.Series(
    log_reg.coef_[0], index=X_bc.columns
).sort_values(key=np.abs, ascending=False)

top5 = coef_series.head(5)
odds_ratios = np.exp(top5)

print('── Top 5 Coefficients (standardised scale) ───────────────────────────')
print(f'{"Feature":<35} {"Coefficient":>12}  {"Odds Ratio":>12}  Interpretation')
print('-' * 85)
for feat, coef in top5.items():
    or_ = np.exp(coef)
    direction = 'increases' if or_ > 1 else 'decreases'
    print(f'{feat:<35} {coef:>12.4f}  {or_:>12.4f}  '
          f'1-unit ↑ {direction} odds of benign by {abs(or_-1)*100:.1f}%')

## Section 7 — Regularization: L1 and L2

When a model has many features (especially correlated ones), coefficients can become very large, leading to overfitting. **Regularization** adds a penalty on large coefficients.

| Method | Penalty | Effect |
|--------|---------|--------|
| **Ridge (L2)** | Sum of squared coefficients | Shrinks all coefficients toward zero |
| **Lasso (L1)** | Sum of absolute coefficients | Drives some coefficients **exactly** to zero (feature selection) |

The `alpha` (or `C = 1/alpha` in sklearn's LogisticRegression) controls strength:
- High alpha → more regularization → simpler model
- Low alpha → less regularization → closer to unregularized

In [ ]:
# Synthetic high-dimensional dataset: 20 features, only 5 are informative
np.random.seed(42)
n_samples, n_features = 150, 20
X_hd = np.random.randn(n_samples, n_features)
true_coef = np.zeros(n_features)
true_coef[:5] = [2.0, -1.5, 1.0, -0.8, 0.5]   # only features 0-4 matter
y_hd = X_hd @ true_coef + np.random.normal(0, 0.5, n_samples)

X_tr_hd, X_te_hd, y_tr_hd, y_te_hd = train_test_split(
    X_hd, y_hd, test_size=0.25, random_state=42
)

# Fit Linear, Ridge, Lasso
models_reg = {
    'Linear (no reg)': LinearRegression(),
    'Ridge (L2, α=1)': Ridge(alpha=1.0),
    'Lasso (L1, α=0.1)': Lasso(alpha=0.1, max_iter=10_000),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (name, model) in zip(axes, models_reg.items()):
    model.fit(X_tr_hd, y_tr_hd)
    test_r2 = model.score(X_te_hd, y_te_hd)

    feat_idx = np.arange(n_features)
    colors_bar = ['tomato' if c != 0 else 'lightgrey' for c in model.coef_]
    ax.bar(feat_idx, model.coef_, color=colors_bar, edgecolor='white')
    ax.plot(feat_idx, true_coef, 'k^', markersize=6, label='True coef', zorder=5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Feature Index')
    ax.set_title(f'{name}\nTest R² = {test_r2:.3f}')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Coefficient Value')
plt.suptitle('Regularization Comparison — 20 Features, Only 5 Are Informative', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Lasso coefficient paths as alpha varies
alphas = np.logspace(-3, 1, 60)
lasso_coefs = []

for a in alphas:
    lasso = Lasso(alpha=a, max_iter=10_000)
    lasso.fit(X_tr_hd, y_tr_hd)
    lasso_coefs.append(lasso.coef_.copy())

lasso_coefs = np.array(lasso_coefs)

plt.figure(figsize=(10, 5))
for i in range(n_features):
    style = '-' if i < 5 else '--'
    lw    = 2.0 if i < 5 else 0.7
    label = f'Feature {i} (informative)' if i < 5 else None
    plt.semilogx(alphas, lasso_coefs[:, i], linestyle=style, linewidth=lw, label=label)

plt.axvline(0.1, color='black', linestyle=':', label='α=0.1 (used above)')
plt.xlabel('Alpha (regularization strength →)')
plt.ylabel('Coefficient')
plt.title('Lasso Coefficient Paths — Features zeroed out as alpha increases')
plt.legend(fontsize=7, ncol=2, loc='upper right')
plt.tight_layout()
plt.show()

# Which features does Lasso select at alpha=0.1?
lasso_sel = Lasso(alpha=0.1, max_iter=10_000)
lasso_sel.fit(X_tr_hd, y_tr_hd)
selected = np.where(lasso_sel.coef_ != 0)[0]
print(f'Features selected by Lasso (α=0.1): {selected.tolist()}')
print(f'True informative features          : {list(range(5))}')

## Section 8 — Comparing Models with Cross-Validation

A single train/test split can be noisy. **K-fold cross-validation** gives a more robust estimate of generalisation performance.

We will compare three logistic regression variants on breast cancer data:
- Logistic Regression (no regularization, `C` very large)
- Ridge Logistic Regression (L2 penalty, `solver='lbfgs'`)
- Lasso Logistic Regression (L1 penalty, `solver='liblinear'`)

In [ ]:
# 5-fold CV comparison on breast cancer data
# StandardScaler is part of each pipeline to avoid data leakage
cv_models = {
    'Logistic (no reg)':  Pipeline([('sc', StandardScaler()),
                                     ('m',  LogisticRegression(C=1e6, max_iter=10_000, random_state=42))]),
    'Ridge Logistic (L2)': Pipeline([('sc', StandardScaler()),
                                      ('m',  LogisticRegression(penalty='l2', C=1.0, max_iter=10_000, random_state=42))]),
    'Lasso Logistic (L1)': Pipeline([('sc', StandardScaler()),
                                      ('m',  LogisticRegression(penalty='l1', C=1.0, solver='liblinear', max_iter=10_000, random_state=42))]),
}

print('── 5-Fold Cross-Validation Accuracy ─────────────────────────────────')
print(f'  {"Model":<30}  {"Mean Acc":>10}  {"Std":>10}')
print('  ' + '-' * 55)

cv_results = {}
for name, pipe in cv_models.items():
    scores = cross_val_score(pipe, X_bc, y_bc, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'  {name:<30}  {scores.mean():>10.4f}  {scores.std():>10.4f}')

# Bar chart of CV scores
fig, ax = plt.subplots(figsize=(8, 4))
names   = list(cv_results.keys())
means   = [v.mean() for v in cv_results.values()]
stds    = [v.std()  for v in cv_results.values()]
colors  = ['steelblue', 'seagreen', 'darkorange']

bars = ax.barh(names, means, xerr=stds, color=colors, alpha=0.8, capsize=4)
ax.set_xlabel('5-Fold CV Accuracy')
ax.set_title('Model Comparison: 5-Fold Cross-Validation')
ax.set_xlim(0.9, 1.0)
for bar, mean in zip(bars, means):
    ax.text(mean + 0.001, bar.get_y() + bar.get_height()/2,
            f'{mean:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

## Section 9 — Model Interpretation

A model that you cannot explain is hard to trust and hard to debug. There are two main ways to interpret a logistic regression model:

1. **Coefficients** — after standardisation, larger |coefficient| → more influential feature.
2. **Permutation importance** — shuffle one feature at a time, measure accuracy drop.

We focus on the standardised coefficient approach here.

In [ ]:
# Fit the best model (Ridge L2) on the full training set
best_pipe = Pipeline([
    ('sc', StandardScaler()),
    ('m',  LogisticRegression(penalty='l2', C=1.0, max_iter=10_000, random_state=42))
])
best_pipe.fit(X_tr_bc, y_tr_bc)

# Extract standardised coefficients
coefs     = best_pipe.named_steps['m'].coef_[0]
feat_names = list(X_bc.columns)

coef_df = pd.DataFrame({
    'feature': feat_names,
    'coef':    coefs,
    'abs_coef': np.abs(coefs)
}).sort_values('abs_coef', ascending=True).tail(15)

plt.figure(figsize=(9, 6))
bar_colors = ['tomato' if c < 0 else 'steelblue' for c in coef_df['coef']]
plt.barh(coef_df['feature'], coef_df['coef'], color=bar_colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Standardised Coefficient')
plt.title('Top 15 Features by Absolute Coefficient\n(blue = increases P(benign), red = decreases P(benign))')
plt.tight_layout()
plt.show()

print('Note: coefficients are on standardised scale — comparable across features.')
print('Positive coefficient → feature increases probability of being BENIGN.')
print('Negative coefficient → feature increases probability of being MALIGNANT.')

In [ ]:
# Permutation importance concept
# (Runs on test set — shows actual impact on accuracy)
X_te_bc_arr = X_te_bc.values if hasattr(X_te_bc, 'values') else X_te_bc
perm = permutation_importance(
    best_pipe, X_te_bc, y_te_bc,
    n_repeats=20, random_state=42, scoring='accuracy'
)

perm_df = pd.DataFrame({
    'feature': feat_names,
    'importance_mean': perm.importances_mean,
    'importance_std':  perm.importances_std
}).sort_values('importance_mean', ascending=True).tail(15)

plt.figure(figsize=(9, 6))
plt.barh(
    perm_df['feature'], perm_df['importance_mean'],
    xerr=perm_df['importance_std'], color='seagreen', alpha=0.8, capsize=3
)
plt.xlabel('Mean Accuracy Drop when Feature is Shuffled')
plt.title('Permutation Feature Importance (Top 15)')
plt.tight_layout()
plt.show()

print('Permutation importance is model-agnostic and reflects true predictive value.')
print('A large accuracy drop = the model relies heavily on that feature.')

## Section 10 — Putting It Together: Model Report

In a real project you want a **reusable evaluation function** that prints all key metrics at once. Here we define `model_report()` and run it on our best breast cancer model.

In [ ]:
from sklearn.metrics import roc_curve, auc

def model_report(model, X_test, y_test, target_names=None):
    """Print a comprehensive classification model report."""
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None

    acc  = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
    auc_roc = roc_auc_score(y_test, y_proba) if y_proba is not None else None

    print('=' * 55)
    print('  MODEL REPORT')
    print('=' * 55)
    print(f'  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)')
    if auc_roc:
        print(f'  AUC-ROC   : {auc_roc:.4f}')

    for label, metrics in report.items():
        if isinstance(metrics, dict):
            print(f'\n  Class: {label}')
            print(f'    Precision : {metrics["precision"]:.4f}')
            print(f'    Recall    : {metrics["recall"]:.4f}')
            print(f'    F1-Score  : {metrics["f1-score"]:.4f}')
            print(f'    Support   : {int(metrics["support"])}')

    print('=' * 55)

    # ROC curve
    if y_proba is not None:
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, color='steelblue', linewidth=2,
                 label=f'ROC curve (AUC = {auc_roc:.3f})')
        plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
        plt.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curve')
        plt.legend()
        plt.tight_layout()
        plt.show()

# Run on best model
model_report(best_pipe, X_te_bc, y_te_bc, target_names=bc.target_names)

### Summary: What We Built

```
  Dataset              Model                 Key Metric
  ───────────────────────────────────────────────────────────
  tips (regression)    Linear Regression     R² ≈ 0.43
  tips (regression)    Polynomial deg=2      R² ≈ 0.44 (marginal gain)
  breast cancer        Logistic Regression   Accuracy > 97%
  breast cancer        Ridge Logistic (L2)   Accuracy > 97%, most stable
  synthetic            Lasso (L1)            Correctly selects 5/20 features
```

**Rules of thumb:**
1. Start simple. Beat the baseline before adding complexity.
2. Use cross-validation to compare models fairly.
3. Apply regularization when you have many features or correlated predictors.
4. Lasso for feature selection; Ridge to stabilise correlated predictors.
5. Always interpret coefficients in the context of the feature's units and scale.

## Section 11 — Practice Exercises

Work through the three exercises below. Each has a blank cell for your solution, followed by a collapsed solution cell.

---

### Exercise 1: Polynomial Regression on the MPG Dataset

Load `seaborn.load_dataset('mpg')`, drop rows with missing values, and:
1. Predict `mpg` from `horsepower`.
2. Try polynomial degrees 1, 2, and 3.
3. Use 5-fold CV to find the best degree.
4. Plot the best fit.

In [ ]:
# Exercise 1 — your solution here


In [ ]:
# Exercise 1 — Solution
mpg = sns.load_dataset('mpg').dropna(subset=['horsepower', 'mpg'])
X_mpg = mpg[['horsepower']].values
y_mpg = mpg['mpg'].values

print('── CV R² by Degree ──────────────────────────')
best_deg, best_score = 1, -np.inf
for deg in [1, 2, 3]:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    scores = cross_val_score(pipe, X_mpg, y_mpg, cv=5, scoring='r2')
    print(f'  Degree {deg}: {scores.mean():.4f} ± {scores.std():.4f}')
    if scores.mean() > best_score:
        best_score, best_deg = scores.mean(), deg

print(f'\nBest degree: {best_deg}')

# Fit and plot
X_tr_m, X_te_m, y_tr_m, y_te_m = train_test_split(X_mpg, y_mpg, test_size=0.2, random_state=42)
best_mpg_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=best_deg, include_bias=False)),
    ('reg',  LinearRegression())
])
best_mpg_pipe.fit(X_tr_m, y_tr_m)

x_r = np.linspace(X_mpg.min(), X_mpg.max(), 300).reshape(-1, 1)
plt.figure(figsize=(7, 4))
plt.scatter(X_mpg, y_mpg, alpha=0.3, s=15, color='steelblue')
plt.plot(x_r, best_mpg_pipe.predict(x_r), color='tomato', linewidth=2,
         label=f'Degree {best_deg} (Test R²={best_mpg_pipe.score(X_te_m, y_te_m):.3f})')
plt.xlabel('Horsepower')
plt.ylabel('MPG')
plt.title('Polynomial Regression: MPG vs Horsepower')
plt.legend()
plt.tight_layout()
plt.show()

---

### Exercise 2: Effect of C (Regularization Strength) on Logistic Regression

Using the breast cancer dataset:
1. Try `C` values: `[0.001, 0.01, 0.1, 1, 10, 100]`.
2. For each C, compute 5-fold CV accuracy.
3. Plot accuracy vs. C on a log scale.
4. What happens at very low / very high C?

In [ ]:
# Exercise 2 — your solution here


In [ ]:
# Exercise 2 — Solution
C_values = [0.001, 0.01, 0.1, 1, 10, 100]
cv_means, cv_stds = [], []

print('── CV Accuracy vs. C ──────────────────────────────────')
for C in C_values:
    pipe = Pipeline([
        ('sc', StandardScaler()),
        ('m',  LogisticRegression(C=C, max_iter=10_000, random_state=42))
    ])
    scores = cross_val_score(pipe, X_bc, y_bc, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())
    print(f'  C={C:<8}  Accuracy = {scores.mean():.4f} ± {scores.std():.4f}')

plt.figure(figsize=(8, 4))
plt.semilogx(C_values, cv_means, 'o-', color='steelblue', linewidth=2)
plt.fill_between(
    C_values,
    np.array(cv_means) - np.array(cv_stds),
    np.array(cv_means) + np.array(cv_stds),
    alpha=0.2, color='steelblue'
)
plt.xlabel('C (inverse regularization strength)')
plt.ylabel('5-Fold CV Accuracy')
plt.title('Logistic Regression: Accuracy vs. C')
plt.tight_layout()
plt.show()

print('\nSmall C = strong regularization (more bias, less variance).')
print('Large C = weak regularization (less bias, potential overfitting).')

---

### Exercise 3: Ridge vs. Lasso Coefficient Sparsity

Using the synthetic high-dimensional dataset (`X_hd`, `y_hd`) from Section 7:
1. Fit Ridge and Lasso for a range of `alpha` values: `np.logspace(-2, 2, 50)`.
2. Count the number of **non-zero** coefficients at each alpha.
3. Plot: number of non-zero coefficients vs. alpha for both methods.
4. At what alpha does Lasso reduce to the true 5 informative features?

In [ ]:
# Exercise 3 — your solution here


In [ ]:
# Exercise 3 — Solution
alphas_ex3 = np.logspace(-2, 2, 50)
ridge_nonzero, lasso_nonzero = [], []

for a in alphas_ex3:
    r = Ridge(alpha=a).fit(X_tr_hd, y_tr_hd)
    l = Lasso(alpha=a, max_iter=10_000).fit(X_tr_hd, y_tr_hd)
    # Ridge never truly zeros out (threshold at 1e-4 for visibility)
    ridge_nonzero.append(np.sum(np.abs(r.coef_) > 1e-4))
    lasso_nonzero.append(np.sum(l.coef_ != 0))

plt.figure(figsize=(9, 4))
plt.semilogx(alphas_ex3, ridge_nonzero, 'o-', color='steelblue', label='Ridge (L2)', markersize=4)
plt.semilogx(alphas_ex3, lasso_nonzero, 's-', color='tomato',    label='Lasso (L1)', markersize=4)
plt.axhline(5, color='black', linestyle='--', linewidth=1, label='True non-zero features = 5')
plt.xlabel('Alpha')
plt.ylabel('Number of Non-Zero Coefficients')
plt.title('Ridge vs. Lasso: Coefficient Sparsity')
plt.legend()
plt.tight_layout()
plt.show()

# Find alpha where Lasso selects exactly 5 features
lasso_exact5 = alphas_ex3[np.array(lasso_nonzero) == 5]
if len(lasso_exact5) > 0:
    print(f'Lasso selects exactly 5 features at alpha ≈ {lasso_exact5[0]:.4f} to {lasso_exact5[-1]:.4f}')
else:
    print('Lasso does not land on exactly 5 in this grid — try a finer search.')

print('\nKey insight: Ridge shrinks but never zeros. Lasso performs automatic feature selection.')